# Generate article embeddings

Generate article-level embeddings for all configured Khmer retrieval models. Run the final cell after reviewing the configuration.

In [1]:
import gc
import json
import logging
from pathlib import Path

import numpy as np

logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
try:
    from transformers.utils import logging as transformers_logging
    transformers_logging.disable_progress_bar()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'search' / 'articles_normalized.jsonl'
OUTPUT_DIR = DATA_DIR / 'processed' / 'embeddings'

MODELS = {
    'sealion': {'model': 'aisingapore/SEA-LION-E5-Embedding-600M', 'provider': 'sentence-transformer', 'batch_size': 8, 'output': 'articles_sealion_e5_600m'},
    'granite-r2': {'model': 'ibm-granite/granite-embedding-311m-multilingual-r2', 'provider': 'sentence-transformer', 'batch_size': 2, 'output': 'articles_granite_311m_r2'},
}
MODEL_KEYS = list(MODELS)

def load_records(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def normalize_vectors(vectors):
    vectors = np.asarray(vectors, dtype='float32')
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError('Embedding output contains a zero vector')
    return vectors / norms

def batches(values, size):
    for start in range(0, len(values), size):
        yield values[start:start + size]

def encode_huggingface(records, model_name, batch_size):
    import torch
    from transformers import AutoModel, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    vectors = []
    with torch.no_grad():
        for batch in batches([r['search_text'] for r in records], batch_size):
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
            output = model(**encoded).last_hidden_state
            mask = encoded['attention_mask'].unsqueeze(-1).float()
            pooled = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
            vectors.append(pooled.cpu().numpy())
    return normalize_vectors(np.concatenate(vectors, axis=0))

def encode_sentence_transformer(records, model_name, batch_size):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    vectors = model.encode([r['search_text'] for r in records], batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    del model
    gc.collect()
    return np.asarray(vectors, dtype='float32')

def write_outputs(records, vectors, config, input_path):
    vectors = np.asarray(vectors, dtype='float32')
    if vectors.ndim != 2 or len(records) != vectors.shape[0] or not np.isfinite(vectors).all():
        raise ValueError('Invalid embedding output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    prefix = config['output']
    np.save(OUTPUT_DIR / f'{prefix}.npy', vectors)
    metadata_path = OUTPUT_DIR / f'{prefix}_metadata.jsonl'
    with metadata_path.open('w', encoding='utf-8') as output:
        for record in records:
            output.write(json.dumps({k: v for k, v in record.items() if k != 'search_text'}, ensure_ascii=False) + '\n')
    manifest = {'model': config['model'], 'provider': config['provider'], 'input_path': str(input_path), 'article_count': len(records), 'embedding_dimension': int(vectors.shape[1]), 'normalized_embeddings': True}
    (OUTPUT_DIR / f'{prefix}_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('Input:', INPUT_PATH)
print('Models:', MODEL_KEYS)

Input: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/search/articles_normalized.jsonl
Models: ['sealion', 'granite-r2']


In [2]:
records = load_records(INPUT_PATH)
results = {}
for model_key, config in MODELS.items():
    if config['provider'] == 'huggingface-mean-pooling':
        vectors = encode_huggingface(records, config['model'], config['batch_size'])
    else:
        vectors = encode_sentence_transformer(records, config['model'], config['batch_size'])
    write_outputs(records, vectors, config, INPUT_PATH)
    results[model_key] = {'records': len(records), 'shape': list(vectors.shape)}
    del vectors
    gc.collect()

print(json.dumps(results, ensure_ascii=False, indent=2))

{
  "sealion": {
    "records": 2744,
    "shape": [
      2744,
      1024
    ]
  },
  "granite-r2": {
    "records": 2744,
    "shape": [
      2744,
      768
    ]
  }
}
